# 1. 피부 분석

In [2]:
# !pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.2/872.2 kB 27.3 MB/s eta 0:00:00


In [8]:
!pip install --upgrade torch torchvision


### (1) 이미지 로드 -> Yolo -> Resnet -> Json

In [2]:
import torch
import cv2
import numpy as np
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn as nn
from torchvision.models import DenseNet201_Weights, VGG19_Weights
import json


# YOLOv5 모델 로드
yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/지우님/yolo_연결/team2_yolov5m_100.pt')
yolo_model.imgsz = 416
yolo_model.conf = 0.3
yolo_model.iou = 0.5
yolo_model.agnostic = True
yolo_model.stride = 1
yolo_model.max_det = 1000
yolo_model.line_thickness = 2

# 데이터 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet50의 입력 크기에 맞게 조정
    transforms.ToTensor(),  # 텐서로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 정규화
])

# 이미지 로드 및 RGB 변환
img_path = '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/예진님/image_detect/Front_data/train/images/0158_03_F.jpg'
img = cv2.imread(img_path)
image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# YOLOv5 모델을 사용하여 탐지
results = yolo_model(image_rgb)

# 자르기
def cropped_img(annotation, resnet_model, results, model_class_id):
    detection_data = results.xyxy[0].cpu().numpy()  # YOLOv5의 탐지 결과 추출
    json_results = []
    # 클래스별로 최대 신뢰도를 추적하기 위한 변수
    max_confidence = -1
    best_box = None
    for *box, conf, cls in detection_data:
        class_id = int(cls)
        if class_id == model_class_id:
            if conf > max_confidence:  # 더 높은 신뢰도일 경우 갱신
                max_confidence = conf
                best_box = box
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        # 바운딩 박스 영역을 자르기
        cropped_img = img[y1:y2, x1:x2]
        # OpenCV 이미지를 PIL 이미지로 변환
        cropped_img_pil = Image.fromarray(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
        # 이미지 전처리
        input_tensor = transform(cropped_img_pil).unsqueeze(0).to(device)
        # 모델에 이미지 입력하여 예측
        with torch.no_grad():
            prediction = resnet_model(input_tensor)
            predicted_class = torch.argmax(prediction, dim=1).item()
        # 결과를 JSON 포맷으로 저장
        json_results.append({
            "class": annotation,
            "predicted_class": predicted_class
        })
    return json_results

##########################################################################################
# skin type

# 클래스 ID와 해당 모델 매핑
skin_type_map = {
    'forehead_skintype': 1,       # 이마
    'l_cheek_skintype' : 5,       # 왼쪽 볼
    'r_cheek_skintype' : 6        # 오른쪽 볼
}

class DenseNetforClassification(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super(DenseNetforClassification, self).__init__()
        self.densenet = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)  # DenseNet121 모델 사용
        self.dropout = nn.Dropout(p=dropout_rate)  # 50% 드롭아웃
        in_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            self.dropout,
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.densenet(x)

skintype_model_settings = {
    'forehead_skintype':{
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/temp/skin_type.pth',
        'num_classes': 2,
        'dropout_rate': 0.4
    },
    'l_cheek_skintype':{
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/temp/skin_type.pth',
        'num_classes': 2,
        'dropout_rate': 0.4
    },
    'r_cheek_skintype':{
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/temp/skin_type.pth',
        'num_classes': 2,
        'dropout_rate': 0.4
    }
}

# 모델 로드 및 예측
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_results = []
skin_type_list =[]
for annotation, class_id in skin_type_map.items():
    model_settings_for_annotation = skintype_model_settings[annotation]
    resnet_model = DenseNetforClassification(
        num_classes=model_settings_for_annotation['num_classes'],
        dropout_rate=model_settings_for_annotation['dropout_rate']
    )
    # state_dict를 로드하고 모델에 적용
    state_dict = torch.load(model_settings_for_annotation['path'], map_location=device)
    resnet_model.load_state_dict(state_dict)
    resnet_model.to(device).eval()
    results_for_annotation = cropped_img(annotation, resnet_model, results, class_id)

    for result in results_for_annotation:
        skin_type_list.append(result['predicted_class'])

# 예측된 클래스를 합산하여 결과를 계산
total_predicted_class = sum(skin_type_list)
if total_predicted_class >= 2:
    json_result_skintype = {
        "class": 'skin_type',
        "predicted_class": 1
    }
else:
    json_result_skintype = {
        "class": 'skin_type',
        "predicted_class": 0
    }

all_results.append(json_result_skintype)

##########################################################################################
# 클래스 ID와 해당 모델 매핑
model_class_id_map = {
    'pigmentation_forehead': 1,  # 이마
    'pigmentation_cheek_l': 5,   # 왼쪽 볼
    'pigmentation_cheek_r': 6,   # 오른쪽 볼
    'wrinkle_perocular_r' : 4,   # 오른쪽 눈가
    'wrinkle_perocular_l' : 3,    # 왼쪽 눈가
    'wrinkle_forehead': 1,       # 이마
    'wrinkle_glabellus': 2,       # 턱
    'chin_sagging' : 8,          # 턱
    'l_cheek_pore' : 5,          # 왼쪽 볼
    'r_cheek_pore' : 6           # 오른쪽 볼
}

#################################################################################################
# 모델 정의
class DenseNet201_VGG19_Ensemble(nn.Module):
    def __init__(self, num_classes, drop_out):
        super(DenseNet201_VGG19_Ensemble, self).__init__()

        # DenseNet201 정의
        self.densenet = models.densenet201(weights=DenseNet201_Weights.DEFAULT)
        densenet_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Identity()  # 최종 분류기를 제거하고 특징만 추출

        # VGG19 정의
        self.vgg = models.vgg19(weights=VGG19_Weights.DEFAULT)
        vgg_features = self.vgg.classifier[0].in_features
        self.vgg.classifier = nn.Identity()  # 최종 분류기를 제거하고 특징만 추출

        # 두 모델의 특징을 결합하는 계층
        self.classifier = nn.Sequential(
            nn.Linear(densenet_features + vgg_features, 1024),
            nn.ReLU(),
            nn.Dropout(p=drop_out),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(p=drop_out),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # DenseNet201 특징 추출
        densenet_features = self.densenet(x)

        # VGG19 특징 추출
        vgg_features = self.vgg(x)

        # 두 특징을 결합
        combined_features = torch.cat((densenet_features, vgg_features), dim=1)

        # 최종 분류
        output = self.classifier(combined_features)
        return output
###############################################################################
# 모델 설정
model_settings = {
    'pigmentation_forehead': {
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/forehead_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pigmentation_cheek_l': {
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pigmentation_cheek_r': {
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },

    'wrinkle_perocular_r': {
        'path':  '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/persocular_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.3
    },
    'wrinkle_perocular_l': {
        'path':   '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/persocular_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.3
    },
    'wrinkle_forehead': {
        'path': '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/forehead_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'wrinkle_glabellus':{
        'path':  '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/wrinkle_glabellus.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'chin_sagging':{
        'path':  '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/chin_sagging.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'l_cheek_pore':{
        'path':  '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pore.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'r_cheek_pore':{
        'path':  '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pore.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    }
}



# 모델 로드 및 예측
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for annotation, class_id in model_class_id_map.items():
    model_settings_for_annotation = model_settings[annotation]
    resnet_model = DenseNet201_VGG19_Ensemble(
        num_classes=model_settings_for_annotation['num_classes'],
        drop_out=model_settings_for_annotation['dropout_rate']

    )
    # state_dict를 로드하고 모델에 적용
    # state_dict의 키 이름을 변경하여 로드
    def rename_state_dict_keys(state_dict, prefix="vgg"):
        renamed_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith("vgg19."):
                new_key = key.replace("vgg19.", f"{prefix}.")
                renamed_state_dict[new_key] = value
            else:
                renamed_state_dict[key] = value
        return renamed_state_dict

# 가중치 로드 및 키 이름 수정
    state_dict = torch.load(model_settings_for_annotation['path'], map_location=device)
    renamed_state_dict = rename_state_dict_keys(state_dict)
    resnet_model.load_state_dict(renamed_state_dict)

    resnet_model.to(device).eval()
    results_for_annotation = cropped_img(annotation, resnet_model, results, class_id)
    all_results.extend(results_for_annotation)
#################################################################################



# JSON 파일로 저장
json_file_path = '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/detection_results.json'
with open(json_file_path, 'w') as f:
    json.dump(all_results, f, indent=2)


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2024-8-31 Python-3.10.12 torch-2.4.0+cu121 CPU

Fusing layers... 
YOLOv5m summary: 212 layers, 20885262 parameters, 0 gradients, 48.0 GFLOPs
Adding AutoShape... 
/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:892: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth
100%|██████████| 77.4M/77.4M [00:00<00:00, 85.8MB/s]
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:06<00:00, 87.1MB/s]


### (2) 이미지 로드 -> Yolo -> Densenet | VGG19 -> Json

In [18]:
import torch
import cv2
import numpy as np
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn as nn
from torchvision.models import DenseNet201_Weights, VGG19_Weights
import json

import pathlib
from pathlib import Path
pathlib.PosixPath = pathlib.WindowsPath


# YOLOv5 모델 로드
yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/team2_yolov5m_100.pt', force_reload=True)
yolo_model.imgsz = 416
yolo_model.conf = 0.3
yolo_model.iou = 0.5
yolo_model.agnostic = True
yolo_model.stride = 1
yolo_model.max_det = 1000
yolo_model.line_thickness = 2

# 데이터 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet50의 입력 크기에 맞게 조정
    transforms.ToTensor(),  # 텐서로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 정규화
])

# 이미지 로드 및 RGB 변환
img_path = 'IMG_9335.JPG'
img = cv2.imread(img_path)
image_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# YOLOv5 모델을 사용하여 탐지
results = yolo_model(image_rgb)
print('Result :', results)

# 자르기
def cropped_img(annotation, resnet_model, results, model_class_id):
    detection_data = results.xyxy[0].cpu().numpy()  # YOLOv5의 탐지 결과 추출
    json_results = []
    # 클래스별로 최대 신뢰도를 추적하기 위한 변수
    max_confidence = -1
    best_box = None
    for *box, conf, cls in detection_data:
        class_id = int(cls)
        if class_id == model_class_id:
            if conf > max_confidence:  # 더 높은 신뢰도일 경우 갱신
                max_confidence = conf
                best_box = box
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        # 바운딩 박스 영역을 자르기
        cropped_img = img[y1:y2, x1:x2]
        # OpenCV 이미지를 PIL 이미지로 변환
        cropped_img_pil = Image.fromarray(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
        # 이미지 전처리
        input_tensor = transform(cropped_img_pil).unsqueeze(0).to(device)
        # 모델에 이미지 입력하여 예측
        with torch.no_grad():
            prediction = resnet_model(input_tensor)
            predicted_class = torch.argmax(prediction, dim=1).item()
        # 결과를 JSON 포맷으로 저장
        json_results.append({
            "class": annotation,
            "predicted_class": predicted_class
        })
    return json_results

##########################################################################################
# 클래스 ID와 해당 모델 매핑
model_class_id_map = {
    'pigmentation_forehead': 1,  # 이마
    'pigmentation_cheek_l': 5,   # 왼쪽 볼
    'pigmentation_cheek_r': 6,   # 오른쪽 볼
    'wrinkle_perocular_r' : 4,   # 오른쪽 눈가
    'wrinkle_perocular_l' : 3,    # 왼쪽 눈가
    'wrinkle_forehead': 1,       # 이마
    'wrinkle_glabellus': 2,       # 턱
    'sagging_chin' : 8,          # 턱
    'pore_cheek_l' : 5,          # 왼쪽 볼
    'pore_cheek_r' : 6           # 오른쪽 볼
}
#################################################################################################
# 모델 정의
class DenseNet201_VGG19_Ensemble(nn.Module):
    def __init__(self, num_classes, drop_out):
        super(DenseNet201_VGG19_Ensemble, self).__init__()

        # DenseNet201 정의
        self.densenet = models.densenet201(weights=DenseNet201_Weights.DEFAULT)
        densenet_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Identity()  # 최종 분류기를 제거하고 특징만 추출

        # VGG19 정의
        self.vgg = models.vgg19(weights=VGG19_Weights.DEFAULT)
        vgg_features = self.vgg.classifier[0].in_features
        self.vgg.classifier = nn.Identity()  # 최종 분류기를 제거하고 특징만 추출

        # 두 모델의 특징을 결합하는 계층
        self.classifier = nn.Sequential(
            nn.Linear(densenet_features + vgg_features, 1024),
            nn.ReLU(),
            nn.Dropout(p=drop_out),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(p=drop_out),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # DenseNet201 특징 추출
        densenet_features = self.densenet(x)

        # VGG19 특징 추출
        vgg_features = self.vgg(x)

        # 두 특징을 결합
        combined_features = torch.cat((densenet_features, vgg_features), dim=1)

        # 최종 분류
        output = self.classifier(combined_features)
        return output
###############################################################################
# 모델 설정
model_settings = {
    'pigmentation_forehead': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/forehead_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pigmentation_cheek_l': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pigmentation_cheek_r': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pigmentation.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },

    'wrinkle_perocular_r': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/persocular_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.3
    },
    'wrinkle_perocular_l': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/persocular_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.3
    },
    'wrinkle_forehead': {
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/forehead_wrinkle.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'wrinkle_glabellus':{
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/wrinkle_glabellus.pth',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'sagging_chin':{
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/chin_sagging.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pore_cheek_l':{
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pore.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    },
    'pore_cheek_r':{
        'path': 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/final_model/cheek_pore.pt',
        'num_classes': 2,
        'dropout_rate': 0.5
    }
}


# 모델 로드 및 예측
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_results = []

for annotation, class_id in model_class_id_map.items():
    model_settings_for_annotation = model_settings[annotation]
    resnet_model = DenseNet201_VGG19_Ensemble(
        num_classes=model_settings_for_annotation['num_classes'],
        drop_out=model_settings_for_annotation['dropout_rate']

    )
    # state_dict를 로드하고 모델에 적용
    # state_dict의 키 이름을 변경하여 로드
    def rename_state_dict_keys(state_dict, prefix="vgg"):
        renamed_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith("vgg19."):
                new_key = key.replace("vgg19.", f"{prefix}.")
                renamed_state_dict[new_key] = value
            else:
                renamed_state_dict[key] = value
        return renamed_state_dict

# 가중치 로드 및 키 이름 수정
    state_dict = torch.load(model_settings_for_annotation['path'], map_location=device)
    renamed_state_dict = rename_state_dict_keys(state_dict)
    resnet_model.load_state_dict(renamed_state_dict)

    resnet_model.to(device).eval()
    results_for_annotation = cropped_img(annotation, resnet_model, results, class_id)
    all_results.extend(results_for_annotation)
#################################################################################


# JSON 파일로 저장
json_file_path = 'detection_results.json'
with open(json_file_path, 'w') as f:
    json.dump(all_results, f, indent=2)


Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\tjdtn/.cache\torch\hub\master.zip
YOLOv5  2024-9-3 Python-3.11.9 torch-2.2.2+cpu CPU

Fusing layers... 
YOLOv5m summary: 212 layers, 20885262 parameters, 0 gradients, 48.0 GFLOPs
Adding AutoShape... 


Result : image 1/1: 3088x2316 1 whole_face, 1 forehead, 2 glabelluss, 2 l_peroculars, 1 l_cheek, 1 r_cheek, 1 lip, 2 chins
Speed: 30.0ms pre-process, 2185.5ms inference, 50.0ms NMS per image at shape (1, 3, 640, 479)


### (3) Json 파일 출력

In [9]:
import json

# JSON 파일을 읽어오는 함수
def load_results_from_json(filename):
    with open(filename, 'r') as json_file:
        results_data = json.load(json_file)
    return results_data

# JSON 파일 경로
json_file_path = 'detection_results.json'


# JSON 파일 읽기
results_data = load_results_from_json(json_file_path)

# 결과 출력
print(json.dumps(results_data, indent=2))

[
  {
    "class": "pigmentation_forehead",
    "predicted_class": 0
  },
  {
    "class": "pigmentation_cheek_l",
    "predicted_class": 1
  },
  {
    "class": "pigmentation_cheek_r",
    "predicted_class": 1
  },
  {
    "class": "wrinkle_perocular_l",
    "predicted_class": 0
  },
  {
    "class": "wrinkle_forehead",
    "predicted_class": 1
  },
  {
    "class": "wrinkle_glabellus",
    "predicted_class": 0
  },
  {
    "class": "sagging_chin",
    "predicted_class": 0
  },
  {
    "class": "pore_cheek_l",
    "predicted_class": 1
  },
  {
    "class": "pore_cheek_r",
    "predicted_class": 1
  }
]


# 2. Json -> LLM : 유저에게 분석 결과 전달

### (1) Python으로 결과값 출력 테스트

In [22]:
import json
import pandas as pd

def analyze_skin_results(json_data):
    # JSON 데이터를 파싱
    results = json.loads(json_data)

    # 분석 결과를 저장할 변수 초기화
    analysis = {
        "pigmentation": [],
        "wrinkle": [],
        "chin": [],
        "pore": []
    }

    # 각 항목을 분석 결과에 분류
    for item in results:
        for key in analysis.keys():
            if key in item["class"]:
                analysis[key].append(item)
                break
            
    # print(analysis)

    # 요약 결과 생성
    # summary = {}
    # for key, items in analysis.items():
    #     if any([x['predicted_class'] == 1 for x in items]):
    #         summary[key] = "관리 필요"
    #     else:
    #         summary[key] = "관리 불필요"

    # # 세부 결과 구성
    # detailed_result = [
    #     f"{item['class'].replace('_', ' ')}: {'추가적인 관리가 필요합니다.' if item['predicted_class'] == 1 else '관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.'}"
    #     for key, items in analysis.items() for item in items
    # ]
    
    detailed_result = {}
    
    for key, items in analysis.items():
        care_annotation = []
        for item in items:
            if item['predicted_class'] == 1:
                if len(item['class'].split('_')[1:]) == 2:
                    if item['class'].split('_')[1:][-1] == 'l':
                        care_annotation.append('left '+ item['class'].split('_')[1:][0])
                    else:
                        care_annotation.append('right '+ item['class'].split('_')[1:][0])
                else:
                    care_annotation.append(item['class'].split('_')[1:][0])
        if len(care_annotation) != 0:
            detailed_result[key] = care_annotation
                
    print(detailed_result.to_frame())         

    # 최종 결과 메시지 생성
    result_message = (
        f"피부 분석 결과:\n"
        f"관리 필요\n"
        f"- 색소침착: {detailed_result['pigmentation']}\n"
        f"- 주름: {detailed_result['wrinkle']}\n"
        # f"- 탄력: {detailed_result['chin']}\n"
        f"- 모공: {detailed_result['pore']}\n\n"
    )

    return result_message

# JSON 파일 경로 설정
json_file_path = 'detection_results.json'

# JSON 파일 로드
with open(json_file_path, 'r') as file:
    json_data = file.read()

# 분석 함수 호출 및 결과 출력
result = analyze_skin_results(json_data)
print(result)


AttributeError: 'dict' object has no attribute 'to_frame'

In [6]:
import json

def analyze_skin_results(json_data):
    # # json 데이터를 불러오기
    results = json.loads(json_data)

    # 분석 결과를 담을 변수를 생성
    analysis = {
        "pigmentation": [],
        "wrinkle": [],
        "chin": [],
        "pore": []
    }

    # 각 부위별로 결과 분석
    for item in results:
        if "pigmentation" in item["class"]:
            analysis["pigmentation"].append(item)
        elif "wrinkle" in item["class"]:
            analysis["wrinkle"].append(item)
        elif "chin" in item["class"]:
            analysis["chin"].append(item)
        elif "pore" in item["class"]:
            analysis["pore"].append(item)

    # 1. 요약 결과 생성
    summary = {
        "pigmentation": "추가적인 관리가 필요합니다." if any([x['predicted_class'] == 1 for x in analysis["pigmentation"]]) else "관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.",
        "wrinkle": "추가적인 관리가 필요합니다." if any([x['predicted_class'] == 1 for x in analysis["wrinkle"]]) else "관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.",
        "chin": "추가적인 관리가 필요합니다." if any([x['predicted_class'] == 1 for x in analysis["chin"]]) else "관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.",
        "pore": "추가적인 관리가 필요합니다." if any([x['predicted_class'] == 1 for x in analysis["pore"]]) else "관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다."
    }

    # 각 피부 부위별 세부 결과 구성
    detailed_result = []
    for key, value in analysis.items():
        for item in value:
            part = item["class"].replace("_", " ")
            status = "추가적인 관리가 필요합니다." if item["predicted_class"] == 1 else "관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다."
            detailed_result.append(f"{part}: {status}")

    # 최종 결과 메시지 생성
    result_message = (
        f"다음은 얼굴 사진을 분석한 결과입니다.:\n"
        f"- 색소 침착은 {summary['pigmentation']}\n"
        f"- 주름은 {summary['wrinkle']}\n"
        f"- 턱 쳐짐은 {summary['chin']}\n"
        f"- 모공은 {summary['pore']}\n\n"
        "세부 분석 결과:\n"
        + "\n".join(detailed_result)
    )

    return result_message

# JSON 파일 경로 설정
json_file_path = '/content/drive/MyDrive/Final_project_2조/02_2. 전처리 및 EDA_이미지/yolo/detection_results.json'

# JSON 파일 로드
with open(json_file_path, 'r') as file:
    json_data = file.read()


# 함수 호출 및 결과 출력
result = analyze_skin_results(json_data)
print(result)

다음은 얼굴 사진을 분석한 결과입니다.:
- 색소 침착은 추가적인 관리가 필요합니다.
- 주름은 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
- 턱 쳐짐은 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
- 모공은 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.

세부 분석 결과:
pigmentation forehead: 추가적인 관리가 필요합니다.
pigmentation cheek l: 추가적인 관리가 필요합니다.
pigmentation cheek r: 추가적인 관리가 필요합니다.
wrinkle perocular r: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
wrinkle perocular l: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
wrinkle forehead: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
wrinkle glabellus: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
chin sagging: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
l cheek pore: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.
r cheek pore: 관리를 잘 하고 계십니다. 지금 하시는 것처럼만 관리해 주시면 됩니다.


### (2) Langchain으로 결과값 출력

In [5]:
import json
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import SystemMessage, HumanMessage
from dotenv import load_dotenv
import os

# .env 파일에서 API 키를 로드
load_dotenv()
# openai_api_key = os.getenv("open_ai_key_team2")

# Langchain의 ChatOpenAI 객체 설정
llm = ChatOpenAI(
    openai_api_key=openai_api_key,
    model_name="gpt-4o-mini",
    temperature=0.56,
)

def create_prompt(json_data):
    # JSON 데이터를 분석 가능한 형태로 정리
    results = json.loads(json_data)

    # 분석 결과를 담을 변수를 초기화
    analysis = {
        "pigmentation": [],
        "wrinkle": [],
        "chin": [],
        "pore": []
    }

    # 각 부위별로 결과를 분석
    for item in results:
        if "pigmentation" in item["class"]:
            analysis["pigmentation"].append(item)
        elif "wrinkle" in item["class"]:
            analysis["wrinkle"].append(item)
        elif "chin" in item["class"]:
            analysis["chin"].append(item)
        elif "pore" in item["class"]:
            analysis["pore"].append(item)

    # 상세 결과를 구성
    detailed_result = []
    for key, value in analysis.items():
        for item in value:
            part = item["class"].replace("_", " ")
            status = "관리 필요" if item["predicted_class"] == 1 else "관리 불필요"
            detailed_result.append(f"{part}: {status}")
            
    # return detailed_result

    # 프롬프트 생성
    # prompt = (
    #     "사용자가 얼굴 사진을 업로드했습니다. 아래는 피부 분석 결과입니다.\n\n"
    #     "피부 분석 결과:\n"
    #     + "\n".join(detailed_result) +
    #     "\n\n위 분석 결과를 문장 형태로 정리해서 사용자에게 전달해주세요."
    #     "\n\n 그리고 결과를 바탕으로 사용자에게 스킨케어 조언을 제공해 주세요."
    #     "추가적인 대화가 필요한 경우, 질문을 이어갈 수 있습니다. 사용자에게 더 자세한 관리 방법이나 제품 추천이 가능하다는 사실을 알려주세요."
    # )
    
    prompt = """
    {detailed_result} is about user's "Skin analysis result". User wants to check "the simplest result".
    
    [Skin analysis result]
    'face skin + state' : 'about care'
    
    [The simplest result]
    title. 피부 진단 결과
    
    content
    1. about care == '관리 필요'인 'face skin + state' 만 알려줘
    
    2. 추가 질문 요청
    '분석 결과에 대한 궁금한 점이 있다면 질문해주세요!
    평소 고민이 있던 부위에 대한 정보를 얻고 싶다면 질문해주세요!'
    """

    return prompt


def get_llm_response(prompt):
    # SystemMessage와 HumanMessage를 사용하여 메시지 설정
    system_message = SystemMessage(content="You are a professional skincare expert.")
    human_message = HumanMessage(content=prompt)

    # LLM 응답 받기
    response = llm([system_message, human_message])
    return response.content

# JSON 파일을 로드합니다.
json_file_path = 'G:/.shortcut-targets-by-id/1RRmm9NKEe67cqs8l-mqiOHyYiPmYkI6y/Final_project_2조/02_2. 전처리 및 EDA_이미지/수린님/detection_results.json'  # JSON 파일의 경로를 입력

with open(json_file_path, 'r') as file:
    json_data = file.read()

# 프롬프트 생성 및 LLM 응답 받기
prompt = create_prompt(json_data)
llm_response = get_llm_response(prompt)

# LLM의 응답 출력
print(llm_response)


### 피부 진단 결과

1. 관리 필요: 'face skin + state'에 해당하는 모든 상태를 확인하였습니다. 

2. 추가 질문 요청: 분석 결과에 대한 궁금한 점이 있다면 질문해주세요! 평소 고민이 있던 부위에 대한 정보를 얻고 싶다면 질문해주세요!


'pigmentation forehead: 관리 불필요',
'pigmentation cheek l: 관리 필요',
'wrinkle perocular l: 관리 불필요', 
'wrinkle forehead: 관리 필요', 
'wrinkle glabellus: 관리 불필요', 
'chin sagging: 관리 불필요', 
'l cheek pore: 관리 필요'